# Beyond the course: TQC + HER solves **PandaPickAndPlace-v3**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elliemci/deep-rl/blob/retrain/panda-tqc-her/units/unit6-a2c-robotics/tqc_her_pick_and_place.ipynb)

Unit 6 trained **A2C** on `PandaPickAndPlace-v3` and it never succeeded (mean reward **-50.00**, the minimum for a 50-step episode).
This notebook trains **TQC + HER** (`sb3-contrib` through RL-Zoo) on the same task and compares both agents on the same metric.
Runtime: **GPU** (Runtime > Change runtime type). The long run is resumable, because everything is written to Google Drive.

| Step | Section |
|---|---|
| Setup, hyperparameters, parameters | 1 to 3 |
| Smoke test (2,000 steps, a few minutes) | 4 |
| Long run, and resume after a disconnect | 5 |
| Evaluation: TQC+HER vs the Unit 6 A2C | 6 and 7 |
| Push to the Hub (only after the results are checked) | 8 |

## Why A2C fails here, and what HER and TQC change

**Sparse reward.** In `PandaPickAndPlace-v3` (sparse reward) the reward is `-1` every step and `0` only when the cube is within 5 cm of the target.
A random policy almost never moves the cube to the target, so almost every episode returns exactly **-50** and carries no learning signal.

**Why on-policy A2C cannot recover.** A2C updates from the last few rollouts and then throws them away. With a sparse reward, a rollout that never
reached the goal produces advantages that are all about the same, so the policy gradient is noise. There is no replay either, so a rare success is used once and forgotten.
Reaching, grasping, lifting and placing is a long chain of actions, so unguided exploration with a few updates per sample has little chance of stumbling on it.

**HER (Hindsight Experience Replay).** A failed episode is still a *successful* episode for a different goal: the place where the cube actually ended up.
HER stores each transition again with the goal replaced by a goal achieved later in the same episode (`goal_selection_strategy="future"`,
`n_sampled_goal=4`), and recomputes the reward for that goal. Failures turn into rewarded examples. This needs an **off-policy** learner and a goal-conditioned
(dict) observation: `observation`, `achieved_goal`, `desired_goal`, which is why the policy is `MultiInputPolicy`.

**TQC (Truncated Quantile Critics).** TQC is SAC with a distributional critic: each critic predicts a set of quantiles of the return, and the
top quantiles are dropped before computing the target. This reduces the overestimation bias that makes plain SAC/DDPG unstable on goal-reaching tasks.
It is off-policy, so every transition (and every relabelled goal) is reused many times, and the GPU is useful for the large 3x512 networks and the 2048 batch.

Formulas and the algorithm descriptions: [HER, Andrychowicz et al. 2017](https://arxiv.org/abs/1707.01495),
[TQC, Kuznetsov et al. 2020](https://arxiv.org/abs/2005.04269), [panda-gym, Gallouedec et al. 2021](https://arxiv.org/abs/2106.13687).

## 1. Setup: Drive, versions, repo files

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Pinned set. rl_zoo3 2.9.2 requires sb3-contrib>=2.9.2, stable-baselines3>=2.9.2 and brings the Hub
# integration (`sb3-team-hf-integration`, imported as `huggingface_sb3`). Do NOT also install the
# separate `huggingface-sb3` package: it provides the same module name and an older version.
# gymnasium is pinned to 1.0.0, the API panda-gym 3.0.7 was written against.
# panda-gym depends on pybullet, which is built from source on Colab (a few minutes the first time).
# Colab's own torch (>=2.8) is kept.
!pip install -q "rl_zoo3==2.9.2" "sb3-contrib==2.9.2" "stable-baselines3==2.9.2" "panda-gym==3.0.7" "gymnasium==1.0.0"

In [ ]:
# The custom hyperparameter file and the launcher live in the GitHub repo.
REPO_BRANCH = "retrain/panda-tqc-her"   # switch to "main" after the branch is merged
import os
if not os.path.exists("/content/deep-rl"):
    !git clone -q --depth 1 -b $REPO_BRANCH https://github.com/elliemci/deep-rl.git /content/deep-rl
UNIT_DIR = "/content/deep-rl/units/unit6-a2c-robotics"
CONF = f"{UNIT_DIR}/configs/tqc_panda_v3.yml"
LAUNCH = f"{UNIT_DIR}/tqc_launch.py"
print(open(CONF).read())

In [ ]:
import importlib.metadata as md, torch, sys
print("python", sys.version.split()[0])
for p in ["rl_zoo3", "sb3-contrib", "stable-baselines3", "sb3-team-hf-integration", "panda-gym", "gymnasium", "pybullet", "torch"]:
    print(f"{p:26s}", md.version(p))
print("CUDA available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
import panda_gym, gymnasium as gym
env = gym.make("PandaPickAndPlace-v3"); print("obs space keys:", list(env.observation_space.spaces)); env.close()

## 2. Hyperparameters

RL-Zoo's `hyperparams/tqc.yml` (v2.9.2) has **no `PandaPickAndPlace-v3` entry**. Its Panda entries are the old **`-v1`** ids (panda-gym 1.x, `gym` API):
`PandaReach-v1` (own tuned entry), and `PandaPush-v1`, `PandaSlide-v1`, `PandaPickAndPlace-v1`, `PandaStack-v1`, which all inherit the shared `&her-defaults`
anchor (defined at `FetchPush-v1`).

I use the **`PandaPickAndPlace-v1` entry, i.e. `her-defaults`, unchanged** and register it under the v3 id in
[`configs/tqc_panda_v3.yml`](https://github.com/elliemci/deep-rl/blob/retrain/panda-tqc-her/units/unit6-a2c-robotics/configs/tqc_panda_v3.yml):
`MultiInputPolicy`, `HerReplayBuffer(goal_selection_strategy="future", n_sampled_goal=4)`, buffer 1M, batch 2048, `gamma=0.95`, `lr=1e-3`, `tau=0.05`,
net `[512, 512, 512]`, 2 critics, 1M steps. The only difference from the Zoo is the environment id (the v1 to v3 dynamics are similar but not identical, so
these values are a starting point and not guaranteed to be tuned for v3).

## 3. Parameters

In [ ]:
N_TIMESTEPS = 1_000_000          # total training steps (the Zoo default for this entry)
SEED        = 0
SAVE_FREQ   = 100_000            # checkpoint + replay buffer every N steps
RUN_DIR     = "/content/drive/MyDrive/ColabNotebooks/DeepRL/tqc_her_runs/pickandplace_seed0"   # on Drive: survives disconnects
REPO_ID     = "elliemci/tqc-PandaPickAndPlace-v3"
PUSH_TO_HUB = False              # keep False until the results are checked
ENV_ID      = "PandaPickAndPlace-v3"
N_EVAL_EPISODES = 100
os.makedirs(RUN_DIR, exist_ok=True)

## 4. Smoke test first (2,000 steps)

Confirms the install, the config, the checkpoint **with replay buffer**, and the resume path before the long run. It writes to a separate `smoke/` folder.

In [ ]:
SMOKE_DIR = f"{RUN_DIR}_smoke"
assert SMOKE_DIR.endswith("_smoke")
!rm -rf "$SMOKE_DIR"      # always start the smoke test from scratch
!python $LAUNCH train --algo tqc --env $ENV_ID --gym-packages panda_gym -conf $CONF \
    -n 2000 --save-freq 1000 --save-replay-buffer --eval-freq -1 -f $SMOKE_DIR --seed $SEED

In [ ]:
# checkpoints AND replay buffers must exist, then a resume must load the 2,000-step buffer (not an empty one)
!ls -la $SMOKE_DIR/tqc/*/
out = !python $LAUNCH resume --run-dir $SMOKE_DIR --target-timesteps 3000 --conf $CONF --seed $SEED --save-freq 1000 -- --eval-freq -1
text = "\n".join(out); print(text[-3000:])
import re
m = re.search(r"REPLAY_BUFFER_SIZE_AFTER_LOAD=(\d+)", text)
assert m and 1900 <= int(m.group(1)) <= 2000, f"resume did not load the 2,000-step replay buffer: {m and m.group(1)}"
print("OK: resumed with a replay buffer of", m.group(1), "transitions")

## 5. Long run

Same command as the RL-Zoo CLI (`rl_zoo3.train`), run through `tqc_launch.py`, a thin wrapper that makes the periodic checkpoints also save the
replay buffer (stock `rl_zoo3.train` only saves it when training ends, which a disconnect would skip) and keeps only the newest two buffers on Drive.
Every checkpoint file is written to local disk first, then copied to Drive, flushed (fsync), test-loaded and only then renamed into place, so a disconnect cannot leave a half-written buffer under its final name.
Evaluation every 50k steps on 20 episodes is logged by RL-Zoo (`eval/success_rate` in TensorBoard / printed output).
Expect roughly 1 to 3 hours on a Colab GPU for 1M steps.

In [ ]:
!python $LAUNCH train --algo tqc --env $ENV_ID --gym-packages panda_gym -conf $CONF \
    -n $N_TIMESTEPS --save-freq $SAVE_FREQ --save-replay-buffer -f $RUN_DIR --seed $SEED \
    --eval-freq 50000 --eval-episodes 20

### Resume after a disconnect

Run sections 1 to 3 again (setup and parameters), **skip** sections 4 and 5 and run this cell. It finds the latest checkpoint **and its own replay buffer** in `RUN_DIR`,
copies both to local disk as `<checkpoint>.zip` + `replay_buffer.pkl` (rl_zoo3 always loads the file named `replay_buffer.pkl` next to the checkpoint, and silently starts with an empty buffer if it is absent),
test-loads the buffer, and **refuses to resume if the buffer is missing, broken or empty**. The log must show `REPLAY_BUFFER_SIZE_AFTER_LOAD=<n>` with n close to the steps done.
It then works out how many steps remain (also across several resumes) and continues in a new experiment folder. The first ~100 steps after a resume use
random actions again (`learning_starts`), a negligible loss.

In [ ]:
!python $LAUNCH resume --run-dir $RUN_DIR --target-timesteps $N_TIMESTEPS --conf $CONF --seed $SEED --save-freq $SAVE_FREQ

## 6. Evaluate over 100 episodes

The main metric is the **success rate**: the fraction of episodes whose last step has `info["is_success"] == True`. Mean reward is secondary
(it is `-(number of steps where the cube is not at the target)`, so -50 means never at the target). The A2C baseline is the Unit 6 agent downloaded from the Hub,
evaluated with the same function and its saved `VecNormalize` statistics.

In [ ]:
import glob, json, numpy as np, gymnasium as gym, panda_gym
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

def evaluate(model, vec_env, n_episodes=N_EVAL_EPISODES, deterministic=True):
    """Roll out `n_episodes` full episodes. Returns success_rate, mean/std reward, mean episode length."""
    obs = vec_env.reset()
    ep_ret, ep_len, rets, succ, lens = 0.0, 0, [], [], []
    while len(rets) < n_episodes:
        action, _ = model.predict(obs, deterministic=deterministic)
        obs, reward, done, info = vec_env.step(action)
        ep_ret += float(reward[0]); ep_len += 1
        if done[0]:
            rets.append(ep_ret); lens.append(ep_len); succ.append(bool(info[0].get("is_success", False)))
            ep_ret, ep_len = 0.0, 0
    return dict(success_rate=float(np.mean(succ)), mean_reward=float(np.mean(rets)), std_reward=float(np.std(rets)),
                mean_ep_len=float(np.mean(lens)), n_episodes=len(rets))

In [ ]:
# --- TQC + HER: final model of the latest experiment folder (falls back to a checkpoint / best model) ---
from sb3_contrib import TQC
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
import glob, os, re, gymnasium as gym

def find_model(run_dir, env_id):
    """Newest experiment folder that has a usable model. Order: final model > latest checkpoint > best_model.zip."""
    exps = sorted(glob.glob(f"{run_dir}/tqc/{env_id}_*"), key=lambda p: int(p.rsplit("_", 1)[1]), reverse=True)
    for exp in exps:
        if os.path.exists(f"{exp}/{env_id}.zip"):
            return exp, f"{exp}/{env_id}.zip", "final model"
        ckpts = sorted(glob.glob(f"{exp}/rl_model_*_steps.zip"), key=lambda p: int(re.search(r"_(\d+)_steps", p).group(1)))
        if ckpts:
            return exp, ckpts[-1], "LATEST CHECKPOINT (the run stopped before saving the final model)"
        if os.path.exists(f"{exp}/best_model.zip"):
            return exp, f"{exp}/best_model.zip", "BEST MODEL by eval success (no final model or checkpoint here)"
    raise FileNotFoundError(f"no model found under {run_dir}/tqc")

exp_dir, model_path, kind = find_model(RUN_DIR, ENV_ID)
print(f"evaluating {model_path}\n  -> {kind}")
tqc_env = DummyVecEnv([lambda: gym.make(ENV_ID)])
if os.path.exists(f"{exp_dir}/{ENV_ID}/vecnormalize.pkl"):          # only if `normalize: True` was added to the yml
    tqc_env = VecNormalize.load(f"{exp_dir}/{ENV_ID}/vecnormalize.pkl", tqc_env)
    tqc_env.training = False
    tqc_env.norm_reward = False
tqc_env.seed(1234)
tqc_model = TQC.load(model_path, env=tqc_env)       # HerReplayBuffer needs an env at load time
tqc_res = evaluate(tqc_model, tqc_env)
print("TQC+HER", tqc_res)

In [ ]:
# --- A2C baseline (Unit 6) from the Hub ---
os.environ["TRUST_REMOTE_CODE"] = "True"   # needed by load_from_hub for the pickled VecNormalize file. Only do this for your OWN repo.
from huggingface_sb3 import load_from_hub
from stable_baselines3 import A2C
a2c_model = A2C.load(load_from_hub("elliemci/a2c-PandaPickAndPlace-v3", "a2c-PandaPickAndPlace-v3.zip"))
a2c_env = DummyVecEnv([lambda: gym.make(ENV_ID)])
a2c_env = VecNormalize.load(load_from_hub("elliemci/a2c-PandaPickAndPlace-v3", "vec_normalize.pkl"), a2c_env)
a2c_env.training = False; a2c_env.norm_reward = False
a2c_env.seed(1234)
a2c_res = evaluate(a2c_model, a2c_env)
print("A2C    ", a2c_res)

In [ ]:
from IPython.display import Markdown
rows = [("A2C (Unit 6, course)", "on-policy, VecNormalize, sparse reward", a2c_res),
        ("TQC + HER (this notebook)", f"off-policy, HER future x4, {N_TIMESTEPS:,} steps, seed {SEED}", tqc_res)]
table = "| agent | training | success rate | mean reward +/- std | mean episode length |\n|---|---|---|---|---|\n" + "\n".join(
    f"| {n} | {t} | **{r['success_rate']:.0%}** | {r['mean_reward']:.2f} +/- {r['std_reward']:.2f} | {r['mean_ep_len']:.1f} |" for n, t, r in rows)
json.dump({"a2c": a2c_res, "tqc_her": tqc_res, "n_timesteps": N_TIMESTEPS, "seed": SEED}, open(f"{RUN_DIR}/eval_results.json", "w"), indent=1)
Markdown(table + f"\n\n{N_EVAL_EPISODES} episodes each, deterministic actions, `PandaPickAndPlace-v3`, same eval seed.")

## 7. Results

1M steps, seed 0, 100 evaluation episodes, deterministic actions, same evaluation seed, `PandaPickAndPlace-v3`.

| agent | training | success rate | mean reward +/- std | mean episode length |
|---|---|---|---|---|
| A2C (Unit 6, course) | on-policy, VecNormalize | 5% | -47.69 +/- 10.21 | 47.7 |
| **TQC + HER** | off-policy, HER (future x4), 1M steps | **36%** | **-34.08 +/- 21.30** | 34.4 |

**Read with care.** This is one seed and 100 episodes, so 36% has roughly a +/- 9 point 95% interval and 5% roughly 2 to 11%.
The gap is large, but the exact numbers are not precise. The task is **not solved**: 64% of episodes still fail.
The A2C 5% is probably not "A2C learned to pick and place". It is most likely episodes where the target happens to spawn right next to the cube
(a hypothesis I have not checked, for example by logging the cube-to-target distance at reset).

**Learning curve.** Success rate over 20 evaluation episodes every 50k steps (noisy: one 20-episode sample per point). The last two points (950k, 1M) come from the resumed run.

In [ ]:
import matplotlib.pyplot as plt
EVAL_CURVE = [0, 25, 10, 10, 30, 30, 10, 30, 20, 10, 30, 35, 25, 40, 55, 25, 40, 55, 35, 25]   # % success, copied from the run's eval log
steps = [50_000 * (i + 1) for i in range(len(EVAL_CURVE))]
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(steps, EVAL_CURVE, marker="o", color="#2a6fdb", label="TQC+HER, 20 episodes per point")
ax.axhline(36, color="#2a6fdb", ls="--", lw=1, label="final 100-episode evaluation: 36%")
ax.axhline(5, color="#c0392b", ls="--", lw=1, label="A2C (Unit 6): 5%")
ax.set_xlabel("training steps"); ax.set_ylabel("success rate (%)"); ax.set_ylim(0, 100)
ax.set_title("PandaPickAndPlace-v3: eval success during training"); ax.grid(alpha=0.3); ax.legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

**Next:** resume the same run to 1.5M to 2M steps (same config, same seed) and re-evaluate with the identical protocol.
Other ideas, each as a separate run so this baseline stays reproducible: `normalize: True` in its own config and run folder, other seeds, `n_sampled_goal` / `gamma` tuning.

## 8. Push to the Hub (only after the results are confirmed)

Uploads the model, config, model card and a **replay video** to `elliemci/tqc-PandaPickAndPlace-v3`. Keep `PUSH_TO_HUB = False` until the table above is accepted.

* `notebook_login()` can fail to prompt in Colab, so login is a separate cell with `getpass`, checked with `whoami()`.
* `rl_zoo3.push_to_hub` needs the `huggingface_sb3` module from **`sb3-team-hf-integration` >= 3**. If the old `huggingface-sb3` package is installed it shadows it and the push fails with
  `ImportError: cannot import name 'generate_metadata'`. The version check below reports that, and the cell falls back to `huggingface_sb3.package_to_hub`.

In [ ]:
from getpass import getpass
from huggingface_hub import login, whoami
if PUSH_TO_HUB:
    login(token=getpass("Hugging Face token with WRITE access: "))
    print("logged in as:", whoami()["name"])          # must be the account that owns REPO_ID

In [ ]:
import importlib.metadata as md
def _ver(pkg):
    try: return md.version(pkg)
    except md.PackageNotFoundError: return None
print("sb3-team-hf-integration:", _ver("sb3-team-hf-integration"), "| huggingface-sb3 (should be None):", _ver("huggingface-sb3"))
if _ver("huggingface-sb3"):
    print("WARNING: the old `huggingface-sb3` package is installed and shadows the one rl_zoo3 needs. Fix: "
          "!pip uninstall -y huggingface-sb3, then !pip install -q --force-reinstall --no-deps sb3-team-hf-integration==3.2, then restart the runtime.")

if PUSH_TO_HUB:
    assert whoami()["name"] == REPO_ID.split("/")[0], "logged in as a different user than the repo owner"
    org, name = REPO_ID.split("/")
    msg = "TQC+HER on PandaPickAndPlace-v3"
    r = !python $LAUNCH push --env $ENV_ID --algo tqc -f $RUN_DIR -orga $org -name $name -n 300 -m "$msg"
    print("\n".join(r[-15:]))
    if any("ImportError" in line or "Traceback" in line for line in r):
        print("rl_zoo3.push_to_hub failed -> falling back to huggingface_sb3.package_to_hub")
        from huggingface_sb3 import package_to_hub
        package_to_hub(model=tqc_model, model_name=name, model_architecture="TQC", env_id=ENV_ID,
                       eval_env=tqc_env, repo_id=REPO_ID, commit_message=msg, is_deterministic=True, n_eval_episodes=10)
else:
    print("PUSH_TO_HUB is False: nothing was uploaded.")

## 9. Optional: unattended finish (resume, summarise, shut the runtime down)

Set `AUTO_SHUTDOWN = True` and run this one cell **instead of queuing cells** (queued cells do not survive a browser disconnect). It resumes training to `N_TIMESTEPS`,
prints a summary, flushes Google Drive and releases the runtime so it stops using compute units. Nothing is pushed to the Hub.

In [ ]:
AUTO_SHUTDOWN = False      # True = after the summary, flush Drive and release the Colab runtime
if AUTO_SHUTDOWN:
    import subprocess, sys, numpy as np
    sys.path.insert(0, UNIT_DIR); import tqc_launch as tl
    subprocess.run([sys.executable, LAUNCH, "resume", "--run-dir", RUN_DIR, "--target-timesteps", str(N_TIMESTEPS),
                    "--conf", CONF, "--seed", str(SEED), "--save-freq", str(SAVE_FREQ)], check=False)
    print("\n=== SUMMARY ===")
    print(f"steps done: {tl.steps_done(RUN_DIR):,} / {N_TIMESTEPS:,}")
    for exp in tl.exp_dirs(RUN_DIR):
        f = exp / "evaluations.npz"
        if f.exists():
            d = np.load(f); succ = d["successes"].mean(axis=1) * 100 if "successes" in d else []
            print(exp.name, "eval success % per 50k:", [int(x) for x in succ])
    from google.colab import drive, runtime
    drive.flush_and_unmount()
    print("Drive flushed. Releasing the runtime.")
    runtime.unassign()